# 🚗 JalanCerdas AI — Colab Training Pipeline

YOLOv8n pothole/road damage detection. Run all cells top-to-bottom.

**Prereq:** Runtime → Change runtime type → **T4 GPU**

Dataset: https://www.kaggle.com/datasets/habibiahmadim09/kerusakan-jalan

## 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics

import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  NO GPU! Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("No GPU detected")

## 2 — Download Dataset from Kaggle

Upload your `kaggle.json` when prompted.

**How to get kaggle.json:**
1. Go to https://www.kaggle.com/settings → API → Create New Token
2. Download `kaggle.json`
3. Upload it in the next cell

In [ ]:
!pip install -q kaggle

from google.colab import files
import os, json, yaml
from pathlib import Path

# Upload kaggle.json
print("Upload kaggle.json (from Kaggle Settings → API → Create New Token):")
uploaded = files.upload()

# Setup credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle credentials configured")

In [ ]:
# Download dataset
!kaggle datasets download -d habibiahmadim09/kerusakan-jalan -p /content/raw_dataset --unzip

RAW_DIR = Path("/content/raw_dataset")
print(f"\nDownloaded to: {RAW_DIR}")

# Show directory structure
print("\n📁 Directory structure:")
for p in sorted(RAW_DIR.rglob("*")):
    if p.is_file():
        depth = len(p.relative_to(RAW_DIR).parts) - 1
        print(f"  {'  ' * depth}{p.name}  ({p.stat().st_size / 1024:.0f} KB)")
    elif p.is_dir() and any(p.iterdir()):
        depth = len(p.relative_to(RAW_DIR).parts)
        n_files = sum(1 for _ in p.iterdir())
        print(f"  {'  ' * depth}{p.name}/  ({n_files} items)")

## 3 — Prepare Dataset (Auto-detect format & convert to YOLO)

This cell auto-detects the dataset format (YOLO / COCO / VOC) and converts to the standard YOLO structure needed for training.

In [ ]:
import shutil, random, json
import xml.etree.ElementTree as ET
from collections import defaultdict

DATASET_DIR = Path("/content/dataset")
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42

# --- Helper functions ---
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_images(d):
    imgs = []
    for ext in IMG_EXTS:
        imgs.extend(d.rglob(f"*{ext}"))
        imgs.extend(d.rglob(f"*{ext.upper()}"))
    return sorted(set(imgs))

def find_labels(d):
    return sorted(d.rglob("*.txt"))

def detect_format(src):
    """Auto-detect annotation format."""
    labels = find_labels(src)
    if labels:
        # Check if YOLO format (5 values per line)
        for lbl in labels[:5]:
            with open(lbl) as f:
                line = f.readline().strip()
                parts = line.split()
                if len(parts) == 5:
                    return "yolo"
        return "txt_other"

    # Check for COCO JSON
    for jf in src.rglob("*.json"):
        try:
            data = json.load(open(jf))
            if "annotations" in data and "images" in data:
                return "coco"
        except:
            pass

    # Check for Pascal VOC XML
    if list(src.rglob("*.xml")):
        return "voc"

    return "images_only"

def coco_to_yolo(coco_json, out_labels):
    with open(coco_json) as f:
        coco = json.load(f)
    cats = {c["id"]: idx for idx, c in enumerate(coco["categories"])}
    imgs = {i["id"]: i for i in coco["images"]}
    img_anns = defaultdict(list)
    for a in coco["annotations"]:
        img_anns[a["image_id"]].append(a)
    for img_id, anns in img_anns.items():
        img = imgs[img_id]
        w, h = img["width"], img["height"]
        lines = []
        for a in anns:
            if "bbox" not in a:
                continue
            x, y, bw, bh = a["bbox"]
            cx = (x + bw/2) / w
            cy = (y + bh/2) / h
            nw = bw / w
            nh = bh / h
            lines.append(f"{cats[a['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        if lines:
            lbl = out_labels / (Path(img["file_name"]).stem + ".txt")
            lbl.write_text("\n".join(lines))
    return cats

def voc_to_yolo(xml_file, out_labels, class_map):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    w = int(root.find("size/width").text)
    h = int(root.find("size/height").text)
    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name not in class_map:
            class_map[name] = len(class_map)
        bbox = obj.find("bndbox")
        xmin, ymin = float(bbox.find("xmin").text), float(bbox.find("ymin").text)
        xmax, ymax = float(bbox.find("xmax").text), float(bbox.find("ymax").text)
        cx = ((xmin + xmax) / 2) / w
        cy = ((ymin + ymax) / 2) / h
        bw = (xmax - xmin) / w
        bh = (ymax - ymin) / h
        lines.append(f"{class_map[name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    if lines:
        lbl = out_labels / (Path(xml_file).stem + ".txt")
        lbl.write_text("\n".join(lines))


# === DETECT FORMAT ===
# Find the actual data root (might be nested inside a subdirectory)
data_root = RAW_DIR
# Look for common subdirs: images/, data/, train/, etc.
for candidate in ["images", "data", "train", "valid", "test"]:
    sub = data_root / candidate
    if sub.exists() and sub.is_dir() and any(sub.iterdir()):
        # Check if this has images
        if find_images(sub):
            data_root = sub.parent
            break

# Also check one level deeper (common kaggle pattern: dataset-name/dataset-name/...)
if not find_images(data_root) and not list(data_root.rglob("*.json")) and not list(data_root.rglob("*.xml")):
    subdirs = [d for d in data_root.iterdir() if d.is_dir()]
    if len(subdirs) == 1:
        data_root = subdirs[0]

fmt = detect_format(data_root)
print(f"Detected format: {fmt}")
print(f"Data root: {data_root}")

# Count images
all_images = find_images(data_root)
print(f"Total images found: {len(all_images)}")

# === BUILD IMAGE-LABEL PAIRS ===
pairs = []  # (image_path, label_text_or_None)

if fmt == "yolo":
    print("\n✅ YOLO format detected — using directly")
    for img in all_images:
        # Find matching label
        lbl = None
        for candidate in [
            img.parent.parent / "labels" / (img.stem + ".txt"),
            img.parent / "labels" / (img.stem + ".txt"),
            img.with_suffix(".txt"),
            img.parent / (img.stem + ".txt"),
        ]:
            if candidate.exists():
                lbl = candidate
                break
        pairs.append((img, lbl))

elif fmt == "coco":
    print("\n🔄 COCO format — converting to YOLO...")
    # Find COCO JSON
    for jf in data_root.rglob("*.json"):
        data = json.load(open(jf))
        if "annotations" in data:
            out_labels = Path("/content/coco_labels")
            out_labels.mkdir(exist_ok=True)
            cats = coco_to_yolo(jf, out_labels)
            print(f"  Classes: {cats}")
            break
    for img in all_images:
        lbl = out_labels / (img.stem + ".txt")
        pairs.append((img, lbl if lbl.exists() else None))

elif fmt == "voc":
    print("\n🔄 Pascal VOC format — converting to YOLO...")
    out_labels = Path("/content/voc_labels")
    out_labels.mkdir(exist_ok=True)
    class_map = {}
    for xml_f in data_root.rglob("*.xml"):
        voc_to_yolo(xml_f, out_labels, class_map)
    print(f"  Classes: {class_map}")
    for img in all_images:
        lbl = out_labels / (img.stem + ".txt")
        pairs.append((img, lbl if lbl.exists() else None))

elif fmt == "images_only":
    print("\n⚠️  Images only — no annotations found!")
    print("You need to annotate images first (Roboflow, CVAT, LabelImg)")
    print("Or check if annotations are in a different format")

else:
    print(f"\n⚠️  Unknown format: {fmt}")
    print("Trying to find any .txt label files...")
    for img in all_images:
        lbl = img.with_suffix(".txt")
        pairs.append((img, lbl if lbl.exists() else None))

# Filter to labeled pairs only
valid_pairs = [(img, lbl) for img, lbl in pairs if lbl is not None]
print(f"\nPairs with labels: {len(valid_pairs)} / {len(pairs)}")

if len(valid_pairs) == 0:
    raise RuntimeError("No labeled images found! Check dataset format.")

In [ ]:
# === SPLIT & ORGANIZE ===
random.seed(SEED)
random.shuffle(valid_pairs)

n = len(valid_pairs)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

splits = {
    "train": valid_pairs[:n_train],
    "val": valid_pairs[n_train:n_train + n_val],
    "test": valid_pairs[n_train + n_val:],
}

# Clean and create structure
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split_name, items in splits.items():
    img_dir = DATASET_DIR / split_name / "images"
    lbl_dir = DATASET_DIR / split_name / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in items:
        # Copy image
        dest_img = img_dir / img_path.name
        shutil.copy2(img_path, dest_img)

        # Copy label
        dest_lbl = lbl_dir / (img_path.stem + ".txt")
        shutil.copy2(lbl_path, dest_lbl)

    print(f"  {split_name}: {len(items)} images")

# === DETECT CLASSES FROM LABELS ===
class_ids = set()
for lbl in (DATASET_DIR / "train" / "labels").glob("*.txt"):
    with open(lbl) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_ids.add(int(parts[0]))

nc = len(class_ids)
max_cls = max(class_ids) if class_ids else 0

# Build class names — use generic names if unknown
# Common road damage classes
ROAD_DAMAGE_NAMES = {
    0: "pothole",
    1: "crack",
    2: "breakage",
    3: "rutting",
    4: "patching",
    5: "void_fill",
    6: "surface_erosion",
}

names = {}
for i in sorted(class_ids):
    names[i] = ROAD_DAMAGE_NAMES.get(i, f"class_{i}")

print(f"\nDetected {nc} classes: {names}")

# If more classes than our mapping, just use generic names
for i in range(max_cls + 1):
    if i not in names:
        names[i] = f"class_{i}"

# === WRITE data.yaml ===
data_cfg = {
    "path": "/content/dataset",
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": nc,
    "names": names,
}

DATA_YAML = "/content/dataset/data.yaml"
with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"\n✅ Dataset ready at {DATASET_DIR}")
print(f"  Train: {len(splits['train'])} images")
print(f"  Val:   {len(splits['val'])} images")
print(f"  Test:  {len(splits['test'])} images")
print(f"  Classes ({nc}): {names}")
print(f"  Config: {DATA_YAML}")

## 4 — Train YOLOv8n

100 epochs, batch 16, image 640. ~15-25 min on T4.

In [ ]:
from ultralytics import YOLO
import time

# --- Config ---
MODEL = "yolov8n.pt"     # nano — fast, small, mobile-friendly
EPOCHS = 100
BATCH = 16
IMGSZ = 640

# --- Load pretrained model ---
model = YOLO(MODEL)

# --- Train ---
start = time.time()
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    patience=30,
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    project="runs",
    name="train_colab",
    exist_ok=True,
    verbose=True,
)

elapsed = time.time() - start
h, rem = divmod(elapsed, 3600)
m, s = divmod(rem, 60)

SAVE_DIR = Path(results.save_dir)
BEST_PT = SAVE_DIR / "weights" / "best.pt"

print(f"\n{'='*50}")
print(f"  Training done in {int(h)}h {int(m)}m {int(s)}s")
print(f"  Save dir:  {SAVE_DIR}")
print(f"  Best PT:   {BEST_PT}")
print(f"  Size:      {BEST_PT.stat().st_size / (1024*1024):.2f} MB")
print(f"{'='*50}")

## 5 — Evaluate Model

In [ ]:
val_model = YOLO(str(BEST_PT))
val_results = val_model.val(data=DATA_YAML, device=0)

print(f"\n📊 Validation Results:")
print(f"  mAP@50:    {val_results.box.map50:.4f}")
print(f"  mAP@50-95: {val_results.box.map:.4f}")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall:    {val_results.box.mr:.4f}")

f1 = 2 * (val_results.box.mp * val_results.box.mr) / (val_results.box.mp + val_results.box.mr + 1e-8)
print(f"  F1 Score:  {f1:.4f}")

## 6 — Export to TFLite

FP16 quantization. Target: <10 MB for mobile.

In [ ]:
export_model = YOLO(str(BEST_PT))

print("📦 Exporting to TFLite (FP16)...")
export_path = export_model.export(
    format="tflite",
    imgsz=IMGSZ,
    half=True,
    simplify=True,
)

tflite_file = Path(export_path)
size_mb = tflite_file.stat().st_size / (1024 * 1024)

print(f"\n✅ TFLite exported!")
print(f"  Path: {tflite_file}")
print(f"  Size: {size_mb:.2f} MB")

if size_mb > 10:
    print(f"  ⚠️  Over 10MB — consider INT8 quant for smaller size")
    print(f"  Re-run: export_model.export(format='tflite', int8=True, data=DATA_YAML)")

## 7 — Test Inference

In [ ]:
import random

# Find test images
test_dir = DATASET_DIR / "test" / "images"
if not test_dir.exists() or not any(test_dir.glob("*.jpg")):
    test_dir = DATASET_DIR / "val" / "images"

all_imgs = list(test_dir.glob("*.jpg")) + list(test_dir.glob("*.png"))
samples = random.sample(all_imgs, min(6, len(all_imgs)))

infer = YOLO(str(BEST_PT))
pred_results = infer.predict(
    source=[str(s) for s in samples],
    conf=0.25,
    device=0,
    save=True,
    project="runs",
    name="predict_colab",
    exist_ok=True,
)

print("\n🔍 Inference results:")
for r, s in zip(pred_results, samples):
    boxes = r.boxes
    n_det = len(boxes)
    confs = [f"{float(c):.0%}" for c in boxes.conf] if n_det else []
    print(f"  {s.name}: {n_det} detection(s) {confs}")

print(f"\nResults saved to: runs/predict_colab/")

## 8 — Download Model

Download `best.pt` and `.tflite` to your local machine.

In [ ]:
from google.colab import files

print(f"⬇️  Downloading best.pt ({BEST_PT.stat().st_size / (1024*1024):.2f} MB)...")
files.download(str(BEST_PT))

if tflite_file.exists():
    print(f"⬇️  Downloading {tflite_file.name} ({size_mb:.2f} MB)...")
    files.download(str(tflite_file))

print("\n✅ Done!")
print("\nNext steps:")
print("  1. Copy .tflite to mobile/assets/models/")
print("  2. Build Flutter APK on local machine")
print("  3. Test camera + GPS + detection")